In [158]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [177]:
# T1
task1 = train.groupby('job')['deposit'].mean().idxmax()

In [176]:
# T2
task2 = train['month'].value_counts().idxmax()
print(task2)

may


In [161]:
# T3
x_train = train.drop(columns=['id', 'deposit'])
y_train = train['deposit']
x_test = test.drop(columns=['id'])

objCols = x_train.select_dtypes(include='object').columns
numCols = x_train.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), objCols),
        ('num', StandardScaler(), numCols)
    ]
)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

gs = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5
)
gs.fit(x_train, y_train)
model = gs.best_estimator_
predictions = model.predict(x_test)

In [164]:
# T4
x_test_processed = preprocessor.fit_transform(x_test)
pca = PCA(n_components=2)
x_test_pca = pca.fit_transform(x_test_processed)
kmeans = KMeans(n_clusters = 2, random_state=42)
clusters = kmeans.fit_predict(x_test_pca)

In [178]:
#DataFrame
rows = []
rows.append({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': task1
})
rows.append({
    'subtaskID': 2,
    'datapointID': 1,
    'answer': task2
})
for id, pred in zip(test['id'], predictions):
    rows.append({
    'subtaskID': 3,
    'datapointID': id,
    'answer': pred
})
for id, cluster in zip(test['id'], clusters):
    rows.append({
    'subtaskID': 4,
    'datapointID': id,
    'answer': cluster
})
subDf = pd.DataFrame(rows)
subDf.to_csv('submission.csv', index=False)